# 16 - Figures: triage first, description second

> **Run order.** Step 16. Needs 04 (figures on disk) and 14 (the `ctx` index). Uses the local GPU
> for about an hour, and is **resumable**: re-running skips every figure already described.
> Logic lives in `src/analyst/vision.py`; see [ADR-010](../docs/adr/0010-figures.md).

The project name says *multimodal*, and until now the 418 extracted figures were stored and
never read. Two measurements came before any code:

- **The images are mostly not charts.** Four samples were a plant photo, blank line art, a CSR
  photo and a QR code.
- **Vector charts are invisible to image extraction**, and counting drawing paths does not find
  them either: the most path-heavy pages were leadership photo grids, icon-laden text pages and
  ruled statement tables.

So a local vision model first labels each image's **kind**, and only chart, table, infographic
and diagram are indexed. The description is generated text, so it is stored in its own table
with the model that wrote it - never in `elements.text`, which stays the filing's own words.

In [1]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 110)

from sqlalchemy import func, select

from analyst import evaluation as ev
from analyst.config import get_settings
from analyst.db import session_scope
from analyst.llm import LLM
from analyst.models import FigureDescription
from analyst.vision import KEEP, describe_all

settings = get_settings()
vision = LLM(settings, "ollama", "gemma3:4b")  # fits in 4 GB VRAM; qwen3-vl (5.7 GB) does not
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
print(vision.name)

ollama/gemma3:4b

## 1. Describe every figure

~8 seconds an image on the RTX 3050. One commit per image, so an interruption costs one image.

**Blank images are never shown to the model.** The first full run found that gemma3:4b turns a
flat white or black image into a revenue chart, numbers included (8 of 12 such images).
`vision.is_blank` now labels them `blank` without a model call. The next cell drops rows written
before that filter existed, so those figures are triaged again. See
[ADR-010](../docs/adr/0010-figures.md), Results.

In [2]:
from pathlib import PureWindowsPath

from analyst.config import ROOT
from analyst.models import ElementRow
from analyst.vision import BLANK, is_blank

with session_scope() as s:
    rows = s.execute(select(FigureDescription, ElementRow.image_path)
                     .join(ElementRow, ElementRow.element_id == FigureDescription.element_id)
                     .where(FigureDescription.kind != BLANK)).tuples().all()
    stale = [d for d, path in rows if is_blank(ROOT / PureWindowsPath(str(path)).as_posix())]
    for d in stale:
        s.delete(d)
print(f"blank figures described before the filter, dropped for re-triage: {len(stale)}")

blank figures described before the filter, dropped for re-triage: 12

In [3]:
import time

t0 = time.perf_counter()
new = describe_all(vision)
print(f"described {sum(new.values())} new figures in {time.perf_counter() - t0:.0f}s")

with session_scope() as s:
    kinds = dict(s.execute(select(FigureDescription.kind, func.count())
                           .group_by(FigureDescription.kind)).tuples().all())
table = pd.Series(kinds, name="figures").sort_values(ascending=False).to_frame()
table["indexed"] = [k in KEEP for k in table.index]
print(table.to_string())
print(f"\nindexed: {table[table.indexed].figures.sum()} of {table.figures.sum()}")

described 12 new figures in 0s

             figures  indexed
photo            236    False
decorative        98    False
qr_code           37    False
logo              19    False
blank             12    False
infographic        8     True
diagram            5     True
chart              3     True


indexed: 16 of 418

## 2. What the model said

Spot-check before trusting it: the kind decides what gets indexed, and every figure in a
description could end up verified as "printed in the evidence" by the agent.

In [4]:
with session_scope() as s:
    rows = s.execute(select(FigureDescription.kind, FigureDescription.element_id,
                            FigureDescription.description)).all()
sample = (pd.DataFrame(rows, columns=["kind", "element", "description"])
          .groupby("kind").head(2).sort_values("kind"))
sample["element"] = sample["element"].str[-24:]
print(sample.to_string(index=False))

       kind                  element                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            description
      blank 025-2a1879ee:p0304:e0005                                                                                                                                                                                                                                                        

## 3. Index the informative figures

Added to the SAME `ctx` collection the agent searches. Point ids are UUIDv5 of the chunk id, so
the existing table and text points are untouched, and a figure chunk can be deleted by type.

In [5]:
from analyst.indexing import load_chunks
from analyst.retrievers import open_store

figs = [c for c in load_chunks(with_context=True, with_figures=True) if c.type == "figure"]
embedder, store = open_store(settings, "bge-small", "ctx")
before = store.count()
if figs:
    store.upsert(figs, list(embedder.embed_documents([c.embed_text for c in figs])))
print(f"figure chunks: {len(figs)}   collection points {before:,} -> {store.count():,}")

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


figure chunks: 16   collection points 10,181 -> 10,197

## 4. Control: did the figures cost the benchmark anything?

Every benchmark answer is a table element. Adding the figure chunks could push some of them down
the ranking. Same questions, same retriever, same collection - now with figures in it.

In [6]:
from analyst.retrievers import dense

search = dense(embedder, store, "ticker+year", expand=True)
cfg = ev.RunConfig(retriever="dense+expand[ctx+figures]", model="bge-small",
                   filters="ticker+year", limit=max(ev.K_VALUES), points=store.count(),
                   notes=f"+{len(figs)} figure chunks described by {vision.name}")
run = ev.build_run(cfg, ev.evaluate(questions, search, max(ev.K_VALUES)), questions,
                   deep=ev.evaluate(questions, search, max(ev.DEPTHS)), root=ev.ROOT)
ev.append_run(run)
ev.write_leaderboard(ev.load_runs())

ledger = [r for r in ev.load_runs()
          if r.config.retriever in ("dense+expand[ctx]", "dense+expand[ctx+figures]")]
print(pd.DataFrame([{**r.row(), **{f"@{d}": v for d, v in r.depth_curve.items()}}
                    for r in ledger]).drop(columns=["model", "filters", "bench"]).to_string())

                                            run                  retriever     R@1     R@3     R@5    R@10     MRR    pR@5  points  p50_ms            git      @1      @5     @10     @20     @50    @100  @200
0          dense+expand[ctx]-bge-small-1ba71392          dense+expand[ctx]  0.1818  0.2273  0.3182  0.4545  0.2428  0.3636   10181   100.7  917bfe7-dirty  0.1818  0.3182  0.4545  0.6364  0.7955  0.8409   1.0
1          dense+expand[ctx]-bge-small-f1b65882          dense+expand[ctx]  0.1818  0.2273  0.3182  0.4545  0.2428  0.3636   10181    91.7  a715497-dirty  0.1818  0.3182  0.4545  0.6364  0.7955  0.8409   1.0
2  dense+expand[ctx+figures]-bge-small-cd581a06  dense+expand[ctx+figures]  0.1818  0.2273  0.3182  0.4545  0.2428  0.3864   10205   104.4        683ea29  0.1818  0.3182  0.4545  0.6364  0.7955  0.8409   1.0
3  dense+expand[ctx+figures]-bge-small-927786e1  dense+expand[ctx+figures]  0.1818  0.2273  0.3182  0.4545  0.2428  0.3864   10197    95.6  683ea29-dirty  0.1818  0.318

## 5. Do figures come back for figure-shaped questions?

**Qualitative only - there is no ground truth for figures**, so this is a look, not a metric.
On the first full run it was misleading: Sun Pharma's top three "revenue trend chart" hits were
blank images with invented descriptions. So the descriptions are printed, not just the ranks.

In [7]:
for q, ticker in [("revenue trend chart over the years", "SUNPHARMA"),
                  ("infographic of key financial highlights", "HDFCBANK"),
                  ("business segments diagram", "RELIANCE")]:
    hits = store.search(embedder.embed_query(q), limit=10, ticker=ticker)
    print(f"{ticker} {q!r}")
    for i, h in enumerate(hits, 1):
        if h.type == "figure":
            print(f"  rank {i:>2}  p{h.pages[0]:<4} {h.text[:90]!r}")

SUNPHARMA 'revenue trend chart over the years'

HDFCBANK 'infographic of key financial highlights'

  rank  1  p80   "This infographic from HDFC Bank's annual report illustrates the 'Impact of Digital Marketi"

  rank  2  p58   'The image shows a table with several circular charts and graphs spread across a desk. One '

RELIANCE 'business segments diagram'

  rank  3  p10   'The infographic displays a map of India with various regions highlighted by orange circles'